# GECKO GPU Benchmark (CUDA vs. WGPU vs. AVX2 CPU)

This notebook builds and benchmarks all three high-speed GECKO backends on real mammalian chromosomes using a free Google Colab **T4 GPU**:
1. **AVX2 CPU (`bin/gecko_fast`)** — single canonical index + streaming prefetch + SIMD extension
2. **Native CUDA + CUB (`bin/gecko_cuda`)** — GPU radix index + lock-free left-maximal filter + warp-cooperative extension
3. **Cross-vendor Rust WGPU (`bin/gecko_wgpu`)** — Vulkan/Metal/DX12 WGSL compute shaders

**Before running:** Select `Runtime -> Change runtime type -> T4 GPU`.

In [ ]:
# 1. Verify NVIDIA GPU and CUDA toolchain
!nvidia-smi
!nvcc --version
!nproc && free -h

In [ ]:
# 2. Clone the repository (update BRANCH to the branch where you pushed these changes)
REPO_URL = "https://github.com/otorreno/gecko.git"
BRANCH = "gpu-backends"  # change to your pushed branch name

!rm -rf gecko
!git clone --depth 1 -b {BRANCH} {REPO_URL} gecko || git clone --depth 1 {REPO_URL} gecko
%cd gecko

In [ ]:
# 3. Build CPU (gecko_fast), CUDA (gecko_cuda), and optionally Rust WGPU (gecko_wgpu)
!make -C src gecko_fast fragStat getInfo
!make -C src gecko_cuda CUDA_ARCH="-gencode arch=compute_75,code=sm_75 -gencode arch=compute_80,code=sm_80"
!./bin/gecko_cuda --check

In [ ]:
# Optional: install Rust and build the cross-vendor WGPU backend (Vulkan on T4)
!which cargo || (curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y)
import os
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]
!make -C src gecko_wgpu
!./bin/gecko_wgpu --check || true

In [ ]:
# 4. Download real mammalian chromosomes (Ensembl GRCh38, gorGor4, Pan_tro_3.0, UCSC mm39)
!mkdir -p /content/mammals
!curl -sL https://ftp.ensembl.org/pub/current_fasta/homo_sapiens/dna/Homo_sapiens.GRCh38.dna_sm.chromosome.21.fa.gz | gunzip > /content/mammals/human_chr21.fa &
!curl -sL https://ftp.ensembl.org/pub/current_fasta/pan_troglodytes/dna/Pan_troglodytes.Pan_tro_3.0.dna_sm.chromosome.21.fa.gz | gunzip > /content/mammals/chimp_chr21.fa &
!curl -sL https://ftp.ensembl.org/pub/current_fasta/gorilla_gorilla/dna/Gorilla_gorilla.gorGor4.dna_sm.chromosome.21.fa.gz | gunzip > /content/mammals/gorilla_chr21.fa &
!curl -sL https://hgdownload.soe.ucsc.edu/goldenPath/mm39/chromosomes/chr16.fa.gz | gunzip > /content/mammals/mouse_chr16.fa &
!curl -sL https://ftp.ensembl.org/pub/current_fasta/homo_sapiens/dna/Homo_sapiens.GRCh38.dna_sm.chromosome.1.fa.gz | gunzip > /content/mammals/human_chr1.fa &
!curl -sL https://ftp.ensembl.org/pub/current_fasta/gorilla_gorilla/dna/Gorilla_gorilla.gorGor4.dna_sm.chromosome.1.fa.gz | gunzip > /content/mammals/gorilla_chr1.fa &
!wait && ls -lh /content/mammals/

In [ ]:
# 5. Head-to-head benchmark + exact fragment parity verifier
import os, struct, subprocess, time
from pathlib import Path

def parse_frags(path):
    frags = []
    with open(path, 'rb') as f:
        n0, n1 = struct.unpack('>QQ', f.read(16))
        while len(b := f.read(93)) == 93:
            diag, xs, ys, xe, ye, length, ident, score, sim, sx, sy, blk, st = struct.unpack('>qQQQQQQQfQQqB', b)
            ch = chr(st)
            if ch == 'r':
                ys = n1 - ys - 1
                ye = n1 - ye - 1
            frags.append((xs, ys, xe, ye, ch, length, score, ident, sx, sy))
    return frags

def check_parity(ref_frags, test_frags, label):
    sr, st = set(ref_frags), set(test_frags)
    exact = len(sr & st)
    by_diag = {}
    for fr in test_frags:
        dk = (fr[4], fr[0]-fr[1] if fr[4]=='f' else fr[0]+fr[1], fr[8], fr[9])
        by_diag.setdefault(dk, []).append((fr[0], fr[2]))
    cov = 0
    for fr in (sr - st):
        dk = (fr[4], fr[0]-fr[1] if fr[4]=='f' else fr[0]+fr[1], fr[8], fr[9])
        for a, b in by_diag.get(dk, []):
            if min(fr[2], b) - max(fr[0], a) + 1 >= 0.9 * (fr[2] - fr[0] + 1):
                cov += 1
                break
    print(f'  [Parity {label}] ref={len(sr):,} test={len(st):,} exact={100*exact/max(len(sr),1):.2f}% recall={100*(exact+cov)/max(len(sr),1):.3f}%')

pairs = [
    ('Human_chr21_vs_Chimp_chr21', '/content/mammals/human_chr21.fa', '/content/mammals/chimp_chr21.fa', 100, 60, 32),
    ('Human_chr21_vs_Gorilla_chr21', '/content/mammals/human_chr21.fa', '/content/mammals/gorilla_chr21.fa', 100, 60, 32),
    ('Human_chr21_vs_Mouse_chr16_WL16', '/content/mammals/human_chr21.fa', '/content/mammals/mouse_chr16.fa', 100, 60, 16),
    ('Human_chr1_vs_Gorilla_chr1_249Mb', '/content/mammals/human_chr1.fa', '/content/mammals/gorilla_chr1.fa', 100, 60, 32),
]

backends = [('cpu', './bin/gecko_fast'), ('cuda', './bin/gecko_cuda')]
if os.path.exists('./bin/gecko_wgpu') and subprocess.run(['./bin/gecko_wgpu', '--check']).returncode == 0:
    backends.append(('wgpu', './bin/gecko_wgpu'))

os.makedirs('/content/out', exist_ok=True)
for name, q, r, lmin, sim, wl in pairs:
    print(f'\n=== {name} (L={lmin}, S={sim}, WL={wl}) ===')
    ref_frags = None
    for bname, bpath in backends:
        outp = f'/content/out/{name}_{bname}'
        t0 = time.perf_counter()
        res = subprocess.run([bpath, q, r, outp, str(lmin), str(sim), str(wl), '1'], capture_output=True, text=True)
        dt = time.perf_counter() - t0
        if res.returncode != 0:
            print(f'  [{bname}] FAILED: {res.stderr.strip()}')
            continue
        print(f'  [{bname}] wall={dt:.3f}s')
        for line in res.stdout.strip().splitlines():
            print('    ' + line)
        frags = parse_frags(outp + '.frags')
        if ref_frags is None:
            ref_frags = frags
        else:
            check_parity(ref_frags, frags, f'{bname} vs cpu')